In [ ]:
import pandas as pd
import numpy as np
# Read in the merged data 
df_clean = pd.read_parquet("../data/processed/df_merged.parquet")

print(df_clean.shape)

c:\Users\Simrat Khalsa\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


(1094411, 24)


In [6]:
missing_summary = pd.DataFrame({
    "missing_count": df_clean.isna().sum(),
    "missing_percent": df_clean.isna().mean() * 100
})

display(
    missing_summary
    .query("missing_count > 0")
    .sort_values("missing_percent", ascending=False)
)

,missing_count,missing_percent
helpfulness,561592,51.314543
review_title,310654,28.385497
hair_color,226768,20.720552
eye_color,209628,19.154413
skin_tone,170539,15.582720
is_recommended,167988,15.349626
tertiary_category,161256,14.734501
skin_type,111557,10.193337
size,43363,3.962223
review_text,1444,0.131943


In [7]:
print("Fully duplicated rows:", df_clean.duplicated().sum())
print("Available columns:", df_clean.columns.tolist())

Fully duplicated rows: 0
Available columns: ['Unnamed: 0', 'author_id', 'rating', 'is_recommended', 'helpfulness', 'total_feedback_count', 'total_neg_feedback_count', 'total_pos_feedback_count', 'submission_time', 'review_text', 'review_title', 'skin_tone', 'eye_color', 'skin_type', 'hair_color', 'product_id', 'product_name', 'brand_name', 'price_usd', 'primary_category', 'secondary_category', 'tertiary_category', 'size', 'loves_count']


In [8]:
blank_text = (
    df_clean["review_text"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
)

print("Missing or blank review text:", blank_text.sum())

Missing or blank review text: 1444


In [9]:
df_clean = df_clean.loc[~blank_text].copy()

print("Remaining reviews:", len(df_clean))

Remaining reviews: 1092967


In [10]:
# Remove reviews with missing or blank text
df_clean = df_clean.loc[~blank_text].copy()

# Inspect rating values
print(df_clean["rating"].value_counts(dropna=False).sort_index())

# Check for invalid ratings
invalid_ratings = ~df_clean["rating"].between(1, 5)

print("Invalid ratings:", invalid_ratings.sum())

rating
1     61105
2     52956
3     81752
4    199211
5    697943
Name: count, dtype: int64
Invalid ratings: 0


In [11]:
# Review length features
df_clean["review_length_words"] = (
    df_clean["review_text"].str.split().str.len()
)

df_clean["review_length_chars"] = (
    df_clean["review_text"].str.len()
)



In [12]:
df_clean[
    ["review_text", "review_length_words", "review_length_chars"]
].head(10)

,review_text,review_length_words,review_length_chars
0,I use this with the Nudestix “Citrus Clean Bal...,79,455
1,I bought this lip mask after reading the revie...,28,162
2,My review title says it all! I get so excited ...,53,272
3,I’ve always loved this formula for a long time...,45,230
4,"If you have dry cracked lips, this is a must h...",46,213
5,The scent isn’t my favourite but it works grea...,36,185
6,I’ll give this 2 stars for nice packaging and ...,64,360
7,I use this at night or while I’m putting makeu...,26,127
8,I love this stuff. I first had the sample size...,55,282
9,I purchased the Sweet Candy scent at my local ...,36,181


In [13]:
df_clean[
    ["review_length_words", "review_length_chars"]
].describe()

,review_length_words,review_length_chars
count,1.092967e+06,1.092967e+06
mean,6.053948e+01,3.205771e+02
std,4.348206e+01,2.314448e+02
min,1.000000e+00,8.000000e+00
25%,3.200000e+01,1.720000e+02
50%,5.000000e+01,2.630000e+02
75%,7.600000e+01,4.020000e+02
max,1.216000e+03,6.448000e+03


In [14]:
import numpy as np

conditions = [
    df_clean["rating"].isin([1, 2]),
    df_clean["rating"].eq(3),
    df_clean["rating"].isin([4, 5])
]

labels = ["Negative", "Neutral/Mixed", "Positive"]

df_clean["rating_group"] = np.select(
    conditions,
    labels,
    default="Unknown"
)

In [15]:
df_clean["rating_group"].value_counts()

rating_group
Positive         897154
Negative         114061
Neutral/Mixed     81752
Name: count, dtype: int64

In [16]:
df_clean[
    [
        "rating",
        "rating_group",
        "review_length_words",
        "review_length_chars"
    ]
].sample(10, random_state=42)

,rating,rating_group,review_length_words,review_length_chars
794989,5,Positive,44,235
564450,5,Positive,67,373
817430,5,Positive,33,204
982191,4,Positive,32,184
760949,4,Positive,40,198
956102,5,Positive,72,369
714689,4,Positive,23,115
4443,5,Positive,36,188
1027502,5,Positive,149,824
662950,5,Positive,27,131


In [17]:
from pathlib import Path

output_path = Path("../data/processed/df_clean.parquet")
output_path.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_parquet(output_path, index=False)

In [18]:
print("Dataset shape:", df_clean.shape)

print("\nReview length statistics:")
display(df_clean["review_length_words"].describe())

print("\nRating group percentages:")
display(
    df_clean["rating_group"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nMissing values in engineered features:")
display(
    df_clean[
        ["review_length_words", "review_length_chars", "rating_group"]
    ].isna().sum()
)

Dataset shape: (1092967, 27)

Review length statistics:


count    1.092967e+06
mean     6.053948e+01
std      4.348206e+01
min      1.000000e+00
25%      3.200000e+01
50%      5.000000e+01
75%      7.600000e+01
max      1.216000e+03
Name: review_length_words, dtype: float64


Rating group percentages:


rating_group
Positive         82.08
Negative         10.44
Neutral/Mixed     7.48
Name: proportion, dtype: float64


Missing values in engineered features:


review_length_words    0
review_length_chars    0
rating_group           0
dtype: int64